# Train U-Net from scratch on Kvasir-SEG

This notebook is a compact training exercise for binary medical-image segmentation. It trains a small U-Net from scratch on Kvasir-SEG and lets you compare a one-GPU run with a two-GPU run inside the same Slurm allocation.

Set `requested_gpus = 1` for the first run, then restart the model and set `requested_gpus = 2` for the second run. The Slurm script reserves two GPUs so both experiments can be done from this Jupyter session.

In [ ]:
from pathlib import Path
import random
import time
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

DATA_ROOT = Path('data')
DATASET_DIR = DATA_ROOT / 'Kvasir-SEG'
ZIP_PATH = DATA_ROOT / 'Kvasir-SEG.zip'
DATA_URL = 'https://datasets.simula.no/kvasir-seg/Kvasir-SEG.zip'

image_size = 256
batch_size = 8
epochs = 5
learning_rate = 1e-3

# Use 1 for the single-GPU exercise, then change to 2 for the two-GPU exercise.
requested_gpus = 1

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPUs:', torch.cuda.device_count())

## Download and inspect the dataset

Kvasir-SEG contains endoscopy images and binary masks for polyp segmentation. If the compute node cannot reach the internet, place `Kvasir-SEG.zip` in `data/` and rerun this cell.

In [ ]:
DATA_ROOT.mkdir(exist_ok=True)

if not DATASET_DIR.exists():
    if not ZIP_PATH.exists():
        print(f'Downloading {DATA_URL}')
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print(f'Extracting {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)

images_dir = DATASET_DIR / 'images'
masks_dir = DATASET_DIR / 'masks'
assert images_dir.exists(), f'Missing {images_dir}'
assert masks_dir.exists(), f'Missing {masks_dir}'

image_files = sorted(images_dir.glob('*'))
mask_files = sorted(masks_dir.glob('*'))
print(f'Images: {len(image_files)}')
print(f'Masks:  {len(mask_files)}')
print('Example image:', image_files[0].name)

In [ ]:
class KvasirSegDataset(Dataset):
    def __init__(self, root, size=256):
        self.root = Path(root)
        self.size = size
        images = {p.stem: p for p in (self.root / 'images').glob('*')}
        masks = {p.stem: p for p in (self.root / 'masks').glob('*')}
        self.keys = sorted(images.keys() & masks.keys())
        self.images = images
        self.masks = masks
        if not self.keys:
            raise RuntimeError('No matching image/mask pairs found')

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, index):
        key = self.keys[index]
        image = Image.open(self.images[key]).convert('RGB').resize((self.size, self.size))
        mask = Image.open(self.masks[key]).convert('L').resize((self.size, self.size), Image.Resampling.NEAREST)

        image = np.asarray(image, dtype=np.float32) / 255.0
        mask = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)

        image = torch.from_numpy(image).permute(2, 0, 1)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask


dataset = KvasirSegDataset(DATASET_DIR, image_size)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(7))

loader_kwargs = dict(batch_size=batch_size, num_workers=4, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

images, masks = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(images[i].permute(1, 2, 0))
    axes[0, i].axis('off')
    axes[1, i].imshow(masks[i, 0], cmap='gray')
    axes[1, i].axis('off')
plt.tight_layout()

## Define U-Net from scratch

The model below uses only randomly initialized convolutional blocks, downsampling, skip connections, and upsampling. There are no pretrained weights.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(channels, feature))
            channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        channels = features[-1] * 2
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(channels, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature * 2, feature))
            channels = feature

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = nn.functional.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat((skip, x), dim=1)
            x = self.ups[i + 1](x)

        return self.final(x)


def dice_score_from_logits(logits, targets, eps=1e-7):
    preds = (torch.sigmoid(logits) > 0.5).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * intersection + eps) / (union + eps)).mean()

In [ ]:
available_gpus = torch.cuda.device_count()
use_gpus = min(requested_gpus, available_gpus)

model = UNet().to(device)
if use_gpus > 1:
    model = nn.DataParallel(model, device_ids=list(range(use_gpus)))
    print(f'Training with DataParallel on {use_gpus} GPUs')
elif use_gpus == 1:
    print('Training on one GPU')
else:
    print('Training on CPU')

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

In [ ]:
def run_epoch(model, loader, training):
    model.train(training)
    total_loss = 0.0
    total_dice = 0.0

    for images, masks in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, masks)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        total_dice += dice_score_from_logits(logits.detach(), masks).item() * images.size(0)

    n = len(loader.dataset)
    return total_loss / n, total_dice / n


history = []
start = time.perf_counter()
for epoch in range(1, epochs + 1):
    train_loss, train_dice = run_epoch(model, train_loader, training=True)
    val_loss, val_dice = run_epoch(model, val_loader, training=False)
    row = dict(epoch=epoch, train_loss=train_loss, train_dice=train_dice, val_loss=val_loss, val_dice=val_dice)
    history.append(row)
    print(f"epoch {epoch:02d} | train loss {train_loss:.4f} dice {train_dice:.4f} | val loss {val_loss:.4f} dice {val_dice:.4f}")

elapsed = time.perf_counter() - start
print(f'Total training time: {elapsed:.1f} seconds')

In [ ]:
model.eval()
images, masks = next(iter(val_loader))
with torch.no_grad():
    logits = model(images.to(device))
preds = (torch.sigmoid(logits).cpu() > 0.5).float()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    axes[0, i].imshow(images[i].permute(1, 2, 0))
    axes[0, i].set_title('image')
    axes[1, i].imshow(masks[i, 0], cmap='gray')
    axes[1, i].set_title('mask')
    axes[2, i].imshow(preds[i, 0], cmap='gray')
    axes[2, i].set_title('prediction')
    for ax in axes[:, i]:
        ax.axis('off')
plt.tight_layout()

## Exercise

1. Run the notebook with `requested_gpus = 1` and record the total training time.
2. Restart the kernel, set `requested_gpus = 2`, and rerun the training cells.
3. Compare speed, validation Dice, and GPU utilization.

For production multi-GPU training, prefer DistributedDataParallel. DataParallel is used here because it is easier to demonstrate interactively inside a notebook.